### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="fiat_500",
    dataset_year="2020",
    domain_str="technology & internet",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/datasets/paolocons/another-fiat-500-dataset-1538-rows",
    download_description="""
kaggle datasets download paolocons/another-fiat-500-dataset-1538-rows -p local-data-warehouse/fiat_500/ && unzip local-data-warehouse/fiat_500/another-fiat-500-dataset-1538-rows.zip -d local-data-warehouse/fiat_500/ && rm local-data-warehouse/fiat_500/another-fiat-500-dataset-1538-rows.zip
""",
    # References
    academic_reference_bibtex=r"""@misc{paolocons2020fiat,
  author       = {Kaggle User Paolocons},
  title        = {Another Dataset on Used Fiat 500 (1538 Rows)},
  year         = {2020},
  howpublished = {\url{https://www.kaggle.com/datasets/paolocons/another-fiat-500-dataset-1538-rows}},
  note         = {Kaggle dataset},
}
""",
    academic_reference_bibtex_key="paolocons2020fiat",
    license="CC0: Public Domain",
    data_tags=["IID"],
    curation_comments="""
- Unlike in TabArena-v0.1, we log scale the price as it is a price and hence log scaling is generally recommended.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="price",
    problem_type="regression",
    objective_metric_name="rmse",
)

## Preprocessing

In [2]:
import pandas as pd
import numpy as np
df = pd.read_csv(f"{dataset_mold.path}/automobile_dot_it_used_fiat_500_in_Italy_dataset_filtered.csv")

cat_features = [
    "model",
]

# Data is ordered, thus dist shift for original order. Shuffling the data removes this.
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df[cat_features] = df[cat_features].astype("category")
df[task_mold.target_column_name] = np.log(df[task_mold.target_column_name])

## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 1,538
Columns: 8
Use sampling: False (sample size: 1,538)
Get row duplicates (staged, merged)...
Using top-7 columns for initial filtering: ['km', 'lon', 'lat', 'age_in_days', 'engine_power', 'previous_owners', 'model']
Rows remaining as candidates after top-7 filter: 43 (of 1,538)

#### Duplicate Report
Total duplicate rows: 18 (1.17% of dataset)
Duplicate rows ignoring target: 23 (1.50% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,model,engine_power,age_in_days,km,previous_owners,lat,lon,price
0,pop,51,3197,120000,2,40.174702,18.167629,8.974618
1,pop,62,2101,103000,1,45.797859,8.644440,8.974618
2,lounge,51,670,32473,1,41.107880,14.208810,9.148465
3,lounge,51,913,29000,1,45.778591,8.946250,9.047821
4,lounge,51,762,18800,1,45.538689,9.928310,9.179881


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,model,category,0.0,0.0,3.0,"lounge, pop, sport"
1,lat,float64,0.0,0.0,449.0,"41.9032, 41.1079, 45.0697, 45.468, 45.4381, 43.7824, 45.5126, 38.1221, 45.5366, 44.5088"
2,lon,float64,0.0,0.0,450.0,"12.4957, 14.2088, 7.7049, 9.1818, 12.3181, 11.255, 10.329, 13.3611, 10.232, 11.4691"
3,price,float64,0.0,0.0,222.0,"9.2591, 9.2965, 9.0938, 9.2003, 9.1485, 9.2873, 9.159, 8.9746, 9.1901, 8.8393"
4,engine_power,int64,0.0,0.0,8.0,"51, 62, 73, 74, 77, 58, 63, 66"
5,age_in_days,int64,0.0,0.0,140.0,"790, 366, 701, 397, 670, 762, 456, 731, 425, 1066"
6,km,int64,0.0,0.0,988.0,"17000, 56779, 15000, 120000, 19000, 100000, 21000, 60000, 90000, 9248"
7,previous_owners,int64,0.0,0.0,4.0,"1, 2, 3, 4"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
engine_power,1538.0,51.904421,3.988023,51.000000,77.000000
age_in_days,1538.0,1650.980494,1289.522278,366.000000,4658.000000
km,1538.0,53396.011704,40046.830723,1232.000000,235000.000000
previous_owners,1538.0,1.123537,0.416423,1.000000,4.000000
lat,1538.0,43.541361,2.133518,36.855839,46.795612
lon,1538.0,11.563428,2.328190,7.245400,18.365520
price,1538.0,9.026199,0.259356,7.824046,9.314700


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column rank                      
model  1     lounge   1094  71.13
       2        pop    358  23.28
       3      sport     86   5.59

In [8]:
# Target Distribution
target_df

,y_missing_count,non_positive_pct,skew_y,skew_log,var_y,var_log,log_used,aic_exponential,aic_lognormal,dist_hint
0,0,0.0,-1.104,-1.162,0.067,0.001,log,23664.1,3.610367e+16,exponential


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=10, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to fiat_500/019d5a4f-cfd0-7f95-843b-6a5809c4dad6
019d5a4f-cfd0-7f95-843b-6a5809c4dad6
6f0f8260d2ff37fcc96eab1f0c8dd25bd280f5a984b6403ab8eaf01b46024513
